# Table of Contents
- [0. The weather-nominal dataset](#The-weather-nominal-dataset)
- [1. Generating frequent patterns with the apriori algorithm](#1.-Generating-frequent-patterns)
- [2. Association rules generation and evaluation](#2.-Association-rules-generation-and-evaluation)


In [1]:
import os
import pandas as pd

# The weather-nominal dataset

Load the weather-nominal dataset: it is an extremely simple dataset with 13 entries and 5 attributes.

In [2]:
data = pd.read_csv('dataset/weather-nominal.csv')
data

,outlook,temperature,humidity,windy,play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes
5,rainy,cool,normal,True,no
6,overcast,cool,normal,True,yes
7,sunny,mild,high,False,no
8,sunny,cool,normal,False,yes
9,rainy,mild,normal,False,yes


Association Rule Mining can be considered as a two-step process:
1. **find all frequent itemsets**: impose a predefined *minimum support* (min sup.).
2. **generate *strong* association rules from the frequent itemsets**: typically, association rules are considered interesting if they satisfy both a *minimum support* threshold and a *minimum confidence* threshold.

# 1. Generating frequent patterns


Frequent pattern analysis is beyond the scope of the `scikit-learn` library. In this notebook we will resort to `mlxtend` ([machine learning extension](http://rasbt.github.io/mlxtend/)), one of the third party libraries that implement the most popular frequent pattern mining algorithms.

In [3]:
from mlxtend.frequent_patterns import apriori

Transform the dataset by setting the column name as a prefix to each value!

In [4]:
for column in data.columns:
    data[column] = data[column].apply(lambda x: f'{column}_{x}')

In [5]:
data

,outlook,temperature,humidity,windy,play
0,outlook_sunny,temperature_hot,humidity_high,windy_False,play_no
1,outlook_sunny,temperature_hot,humidity_high,windy_True,play_no
2,outlook_overcast,temperature_hot,humidity_high,windy_False,play_yes
3,outlook_rainy,temperature_mild,humidity_high,windy_False,play_yes
4,outlook_rainy,temperature_cool,humidity_normal,windy_False,play_yes
5,outlook_rainy,temperature_cool,humidity_normal,windy_True,play_no
6,outlook_overcast,temperature_cool,humidity_normal,windy_True,play_yes
7,outlook_sunny,temperature_mild,humidity_high,windy_False,play_no
8,outlook_sunny,temperature_cool,humidity_normal,windy_False,play_yes
9,outlook_rainy,temperature_mild,humidity_normal,windy_False,play_yes


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   outlook      14 non-null     object
 1   temperature  14 non-null     object
 2   humidity     14 non-null     object
 3   windy        14 non-null     object
 4   play         14 non-null     object
dtypes: object(5)
memory usage: 688.0+ bytes


In [7]:
data.describe()

,outlook,temperature,humidity,windy,play
count,14,14,14,14,14
unique,3,3,2,2,2
top,outlook_sunny,temperature_mild,humidity_high,windy_False,play_yes
freq,5,6,7,8,9


### The apriori algorithm

Apriori is a popular algorithm for extracting frequent itemsets with applications in association rule learning. The apriori algorithm has been designed to operate on databases containing transactions, such as purchases by customers of a store. 

An itemset is considered as "frequent" if it meets a user-specified support threshold. For instance, if the support threshold is set to 0.5 (50%), a frequent itemset is defined as a set of items that occur together in at least 50% of all transactions in the database.



**Encoded format**

The allowed values for a DataFrame provided as input at mlxtend's frequent pattern mining algorithms are either 0/1 or True/False (i.e., boolean vector). 

This encoding complies with the scenario of *market basket analysis*, in which we think of the universe as the set of items available at the store, and each item as a Boolean variable representing the presence or absence of that item. Each basket can be represented by a Boolean vector of values assigned to these variables.

The TransactionEncoder converts item lists into transaction data for frequent itemset mining (simply transforms the input dataset into a one-hot encoded NumPy boolean array)

In [8]:
from mlxtend.preprocessing import TransactionEncoder 
te = TransactionEncoder()
te_data = te.fit(data.values).transform(data.values)
df = pd.DataFrame(te_data, columns = te.columns_)
df

,humidity_high,humidity_normal,outlook_overcast,outlook_rainy,outlook_sunny,play_no,play_yes,temperature_cool,temperature_hot,temperature_mild,windy_False,windy_True
0,True,False,False,False,True,True,False,False,True,False,True,False
1,True,False,False,False,True,True,False,False,True,False,False,True
2,True,False,True,False,False,False,True,False,True,False,True,False
3,True,False,False,True,False,False,True,False,False,True,True,False
4,False,True,False,True,False,False,True,True,False,False,True,False
5,False,True,False,True,False,True,False,True,False,False,False,True
6,False,True,True,False,False,False,True,True,False,False,False,True
7,True,False,False,False,True,True,False,False,False,True,True,False
8,False,True,False,False,True,False,True,True,False,False,True,False
9,False,True,False,True,False,False,True,False,False,True,True,False


Notice that the same encoding can be obtained with `sklearn.preprocessing.OneHotEncoder` and `pandas.get_dummies()`.

Now, obtain the items and itemsets with at least MinSup support (e.g., MinSup = 0.2):



In [9]:
from mlxtend.frequent_patterns import apriori

In [29]:
verbose = False
if verbose:
    apriori?
# - requires setting of min_support
# - requires one-hot encoded dataframe

In [30]:
# get all itemsets with maximum length=1 AND min_support >=0.2
freq_itemset_max1 = apriori(df, 
                       min_support = 0.2, 
                       use_colnames = True,
                       max_len=1,
                       verbose = True)
display(freq_itemset_max1)

,support,itemsets
0,0.500000,(humidity_high)
1,0.500000,(humidity_normal)
2,0.285714,(outlook_overcast)
3,0.357143,(outlook_rainy)
4,0.357143,(outlook_sunny)
5,0.357143,(play_no)
6,0.642857,(play_yes)
7,0.285714,(temperature_cool)
8,0.285714,(temperature_hot)
9,0.428571,(temperature_mild)


In [31]:
# get all itemsetswith min_support >=0.2

freq_itemset = apriori(df, 
                       min_support = 0.2, 
                       use_colnames = True,
                       verbose = True)

Processing 16 combinations | Sampling itemset size 4


In [32]:
freq_itemset

,support,itemsets
0,0.500000,(humidity_high)
1,0.500000,(humidity_normal)
2,0.285714,(outlook_overcast)
3,0.357143,(outlook_rainy)
4,0.357143,(outlook_sunny)
5,0.357143,(play_no)
6,0.642857,(play_yes)
7,0.285714,(temperature_cool)
8,0.285714,(temperature_hot)
9,0.428571,(temperature_mild)


In [33]:
sorted(freq_itemset.support.unique())   #get unique values of support (also, we order them)

[np.float64(0.21428571428571427),
 np.float64(0.2857142857142857),
 np.float64(0.35714285714285715),
 np.float64(0.42857142857142855),
 np.float64(0.5),
 np.float64(0.5714285714285714),
 np.float64(0.6428571428571429)]

In [34]:
[(f'{x} / 14 = {x / 14}') for x in range(3, 10)]       

['3 / 14 = 0.21428571428571427',
 '4 / 14 = 0.2857142857142857',
 '5 / 14 = 0.35714285714285715',
 '6 / 14 = 0.42857142857142855',
 '7 / 14 = 0.5',
 '8 / 14 = 0.5714285714285714',
 '9 / 14 = 0.6428571428571429']

The type of the itemset value is `frozenset`.

In [35]:
freq_itemset.itemsets.values[0]

frozenset({'humidity_high'})

Differently from the classical python `set`, the `frozenset` type is immutable and hashable — its contents cannot be altered after it is created; it can therefore be used as a dictionary key or as an element of another set. 
<br>*Note: sets are unchangeable: once a set is created, you cannot change the elements. BUT: you can remove or add new elements*

We can leverage the power of pandas to conveniently analyse/filter the results. For instance, we can create the DataFrame of frequent itemsets via apriori and then add a new column that stores the length of each itemset:

In [36]:
freq_itemset['length'] = freq_itemset['itemsets'].apply(lambda x: len(x))
freq_itemset

,support,itemsets,length
0,0.500000,(humidity_high),1
1,0.500000,(humidity_normal),1
2,0.285714,(outlook_overcast),1
3,0.357143,(outlook_rainy),1
4,0.357143,(outlook_sunny),1
5,0.357143,(play_no),1
6,0.642857,(play_yes),1
7,0.285714,(temperature_cool),1
8,0.285714,(temperature_hot),1
9,0.428571,(temperature_mild),1


Filter the results based on some desired criteria (e.g., selects only the *k*-itemset with *k*>2 and support>0.25)

In [39]:
freq_itemset[(freq_itemset['length'] > 2) & (freq_itemset['support'] >= 0.25)]

,support,itemsets,length
40,0.285714,"(play_yes, windy_False, humidity_normal)",3


In [45]:
freq_itemset[freq_itemset['itemsets'].apply(lambda x: 'play_no' in x)]

,support,itemsets,length
5,0.357143,(play_no),1
13,0.285714,"(play_no, humidity_high)",2
28,0.214286,"(play_no, outlook_sunny)",2
30,0.214286,"(play_no, windy_True)",2
38,0.214286,"(play_no, humidity_high, outlook_sunny)",3


Note: as we are dealing with frozensets, the order does not matter.

In [41]:
freq_itemset[freq_itemset['itemsets'] == {"play_yes", "outlook_overcast"}]

,support,itemsets,length
24,0.285714,"(play_yes, outlook_overcast)",2


In [42]:
freq_itemset[freq_itemset['itemsets'] == {"outlook_overcast", "play_yes"}]


,support,itemsets,length
24,0.285714,"(play_yes, outlook_overcast)",2


## 2. Association rules generation and evaluation

An association rule is an implication expression of the form $X \rightarrow Y$, where $X$ and $Y$ are disjoint itemsets.

Association rules can be generated as follows:
- for each frequent itemset $l$, generate all nonempty subset of $l$
- for every nonempty subset $s$ of $l$, output the rule "$s \rightarrow (l-s)$" if $\frac{\text{support}(l)}{\text{support}(s)}>\text{min_conf}$

As the rules are generated from frequent itemsets, each one automatically satisfy the *minimum support*.

In the following we generate association rules from the frequent itemsets.

In [43]:
from mlxtend.frequent_patterns import association_rules

In [44]:
association_rules?

In [25]:
df_res = association_rules(freq_itemset, 
                  metric = "confidence", 
                  min_threshold = 0.7)
list_cols = ['antecedents',
 'consequents',
 'antecedent support',
 'consequent support',
 'support',
 'confidence']

df_res[list_cols]


,antecedents,consequents,antecedent support,consequent support,support,confidence
0,(play_no),(humidity_high),0.357143,0.500000,0.285714,0.800000
1,(temperature_hot),(humidity_high),0.285714,0.500000,0.214286,0.750000
2,(humidity_normal),(play_yes),0.500000,0.642857,0.428571,0.857143
3,(temperature_cool),(humidity_normal),0.285714,0.500000,0.285714,1.000000
4,(outlook_overcast),(play_yes),0.285714,0.642857,0.285714,1.000000
5,(temperature_cool),(play_yes),0.285714,0.642857,0.214286,0.750000
6,(windy_False),(play_yes),0.571429,0.642857,0.428571,0.750000
7,(temperature_hot),(windy_False),0.285714,0.571429,0.214286,0.750000
8,"(play_no, humidity_high)",(outlook_sunny),0.285714,0.357143,0.214286,0.750000
9,"(play_no, outlook_sunny)",(humidity_high),0.214286,0.500000,0.214286,1.000000


Let $A \rightarrow C$ be a rule ($A$ and $C$ stands for the antecedent and the consequents, respectively).

The table produced by the association rule mining algorithm contains three different support metrics:
- **antecedent support**: proportion of transactions containing the antecedent $A$  

$\quad \text{support}(A) \quad \text{range:}\; [0,1]$
- **consequent support**:  proportion of transactions containing the consequent $C$ 

$\quad \text{support}(C) \quad \text{range:}\; [0,1]$
- **support**: computes the support of the combined itemset $A \cup C$  

$\quad \text{support}(A\rightarrow C)=\text{support}(A \cup C) \quad \text{range:}\; [0,1]$



Many association rule mining algorithms employ the *support-confidence* framework. Indeed, we find also confidence among the metrics:

- **confidence**: probability of seeing the consequent in a transaction given that it also contains the antecedent. 

$\quad \text{confidence}(A \rightarrow C)  = \frac{\text{support}(A \rightarrow C)}{\text{support}(A)} \quad \text{range:}\; [0,1]$

## Aside

The *support-confidence* framework, however, may still lead to rules that are uninteresting to the users. For example, if $C$ is always (or almost always) present in all the transaction, than the confidence will always be very high, but it will NOT indicate an interesting phenomenon.

<br>
**RARE** items: Or, items that occur very infrequently in the data are pruned although they can be interesting (for example, rare item could be the expensive ones).
<br>
<br>
Different metrics have been developed to supplement such a framework for the evaluation of an association rule. The current implementation of `mlxtend` makes use of confidence, lift, leverage and conviction metrics, thus enabling a *support-confidence-correlation* framework.


- **lift**: how many times more often $A$ and $C$ occur together than expected if they were statistically independent 

$\quad \text{lift}(A \rightarrow C) = \text{lift}(C \rightarrow A) = \frac{\text{confidence}(A \rightarrow C)}{\text{support}(C)} = \frac{\text{confidence}(C \rightarrow A)}{\text{support}(A)}\quad \text{range:}\; [0,\infty]$

- **leverage**: difference between the observed frequency of A and C appearing together and the frequency that would be expected if A and C were independent. A leverage value of 0 indicates independence.

$\quad \text{leverage}(A \rightarrow C) = \text{support}(A \rightarrow C) - \text{support}(A) \times \text{support}(C) \quad \text{range:}\; [-1,1]$

- **conviction**: high conviction value means that the consequent is highly depending on the antecedent. Similar to lift, if items are independent, the conviction is 1.

$\quad \text{conviction}(A \rightarrow C) = \frac{1-\text{support}(C)}{1-\text{confidence}(A \rightarrow C)}  \quad \text{range:}\; [0,\infty]$

- **zhangs_metric**: Measures both association and dissociation. Value ranges between -1 and 1. A positive value (>0) indicates Association and negative value indicated dissociation.

$\quad \text{zhangs_metric}(A \rightarrow C) =  \frac{\text{confidence}(A \rightarrow C) - \text{confidence}(A' \rightarrow C) }{max(\text{confidence}(A \rightarrow C),\text{confidence}(A' \rightarrow C))}
    \quad \text{range:}\; [-1,1]$ 

In [26]:
rules = association_rules(freq_itemset, 
                          metric = "lift", 
                          min_threshold = 1.1)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(humidity_high),(outlook_sunny),0.500000,0.357143,0.214286,0.428571,1.200000,1.0,0.035714,1.125000,0.333333,0.333333,0.111111,0.514286
1,(outlook_sunny),(humidity_high),0.357143,0.500000,0.214286,0.600000,1.200000,1.0,0.035714,1.250000,0.259259,0.333333,0.200000,0.514286
2,(play_no),(humidity_high),0.357143,0.500000,0.285714,0.800000,1.600000,1.0,0.107143,2.500000,0.583333,0.500000,0.600000,0.685714
3,(humidity_high),(play_no),0.500000,0.357143,0.285714,0.571429,1.600000,1.0,0.107143,1.500000,0.750000,0.500000,0.333333,0.685714
4,(humidity_high),(temperature_hot),0.500000,0.285714,0.214286,0.428571,1.500000,1.0,0.071429,1.250000,0.666667,0.375000,0.200000,0.589286
5,(temperature_hot),(humidity_high),0.285714,0.500000,0.214286,0.750000,1.500000,1.0,0.071429,2.000000,0.466667,0.375000,0.500000,0.589286
6,(humidity_high),(temperature_mild),0.500000,0.428571,0.285714,0.571429,1.333333,1.0,0.071429,1.333333,0.500000,0.444444,0.250000,0.619048
7,(temperature_mild),(humidity_high),0.428571,0.500000,0.285714,0.666667,1.333333,1.0,0.071429,1.500000,0.437500,0.444444,0.333333,0.619048
8,(outlook_rainy),(humidity_normal),0.357143,0.500000,0.214286,0.600000,1.200000,1.0,0.035714,1.250000,0.259259,0.333333,0.200000,0.514286
9,(humidity_normal),(outlook_rainy),0.500000,0.357143,0.214286,0.428571,1.200000,1.0,0.035714,1.125000,0.333333,0.333333,0.111111,0.514286


In [27]:
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
0,(humidity_high),(outlook_sunny),0.500000,0.357143,0.214286,0.428571,1.200000,1.0,0.035714,1.125000,0.333333,0.333333,0.111111,0.514286,1
1,(outlook_sunny),(humidity_high),0.357143,0.500000,0.214286,0.600000,1.200000,1.0,0.035714,1.250000,0.259259,0.333333,0.200000,0.514286,1
2,(play_no),(humidity_high),0.357143,0.500000,0.285714,0.800000,1.600000,1.0,0.107143,2.500000,0.583333,0.500000,0.600000,0.685714,1
3,(humidity_high),(play_no),0.500000,0.357143,0.285714,0.571429,1.600000,1.0,0.107143,1.500000,0.750000,0.500000,0.333333,0.685714,1
4,(humidity_high),(temperature_hot),0.500000,0.285714,0.214286,0.428571,1.500000,1.0,0.071429,1.250000,0.666667,0.375000,0.200000,0.589286,1
5,(temperature_hot),(humidity_high),0.285714,0.500000,0.214286,0.750000,1.500000,1.0,0.071429,2.000000,0.466667,0.375000,0.500000,0.589286,1
6,(humidity_high),(temperature_mild),0.500000,0.428571,0.285714,0.571429,1.333333,1.0,0.071429,1.333333,0.500000,0.444444,0.250000,0.619048,1
7,(temperature_mild),(humidity_high),0.428571,0.500000,0.285714,0.666667,1.333333,1.0,0.071429,1.500000,0.437500,0.444444,0.333333,0.619048,1
8,(outlook_rainy),(humidity_normal),0.357143,0.500000,0.214286,0.600000,1.200000,1.0,0.035714,1.250000,0.259259,0.333333,0.200000,0.514286,1
9,(humidity_normal),(outlook_rainy),0.500000,0.357143,0.214286,0.428571,1.200000,1.0,0.035714,1.125000,0.333333,0.333333,0.111111,0.514286,1


In [28]:
rules[(rules['antecedent_len'] >= 2) &
      (rules['confidence'] > 0.75) &
      (rules['lift'] > 1.2) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
31,"(play_no, outlook_sunny)",(humidity_high),0.214286,0.500000,0.214286,1.0,2.000000,1.0,0.107143,inf,0.636364,0.428571,1.0,0.714286,2
32,"(humidity_high, outlook_sunny)",(play_no),0.214286,0.357143,0.214286,1.0,2.800000,1.0,0.137755,inf,0.818182,0.600000,1.0,0.800000,2
37,"(play_yes, temperature_cool)",(humidity_normal),0.214286,0.500000,0.214286,1.0,2.000000,1.0,0.107143,inf,0.636364,0.428571,1.0,0.714286,2
44,"(windy_False, humidity_normal)",(play_yes),0.285714,0.642857,0.285714,1.0,1.555556,1.0,0.102041,inf,0.500000,0.444444,1.0,0.722222,2
48,"(play_yes, outlook_rainy)",(windy_False),0.214286,0.571429,0.214286,1.0,1.750000,1.0,0.091837,inf,0.545455,0.375000,1.0,0.687500,2
50,"(outlook_rainy, windy_False)",(play_yes),0.214286,0.642857,0.214286,1.0,1.555556,1.0,0.076531,inf,0.454545,0.333333,1.0,0.666667,2
